In [ ]:
!pip install requests pandas matplotlib tqdm -q
print('✅ Libraries ready')

✅ Libraries ready


In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import os, zipfile, io, re, math
from tqdm import tqdm

os.makedirs('jeff_data', exist_ok=True)

# Direct download URL for JEFF-4.0 neutron data zip (593 files, 608 MB)
JEFF_ZIP_URL = (
    'https://data.oecd-nea.org/records/e9ajn-a3p20/files/'
    'JEFF40-Evaluations-Neutron-593.zip?download=1'
)

print('✅ Setup complete')
print('📁 Output folder: jeff_data/')

✅ Setup complete
📁 Output folder: jeff_data/


In [ ]:
REACTIONS = [
    {
        'label':   'Fe-56(n,inl)Fe-56star',
        'display': "Fe-56(n,n')Fe-56*",
        'mt':      4,
        'color':   '#9B4CE8',
        'note':    'Total inelastic scattering'
    },
    {
        'label':   'Fe-56(n,2n)Fe-55',
        'display': 'Fe-56(n,2n)Fe-55',
        'mt':      16,
        'color':   '#4CE87B',
        'note':    'Two-neutron emission, threshold ~11.2 MeV'
    },
    {
        'label':   'Fe-56(n,g)Fe-57',
        'display': 'Fe-56(n,γ)Fe-57',
        'mt':      102,
        'color':   '#E84C9B',
        'note':    'Radiative capture'
    },
    {
        'label':   'Fe-56(n,a)Cr-53',
        'display': 'Fe-56(n,α)Cr-53',
        'mt':      107,
        'color':   '#E8D44C',
        'note':    'Alpha emission (ENDF MT=107)'
    },
]

MT_WANTED = {r['mt'] for r in REACTIONS}

print(f'✅ {len(REACTIONS)} reaction channels defined:')
for r in REACTIONS:
    print(f"  • MT={r['mt']:3d}  {r['display']:25s}  [{r['note']}]")

✅ 4 reaction channels defined:
  • MT=  4  Fe-56(n,n')Fe-56*          [Total inelastic scattering]
  • MT= 16  Fe-56(n,2n)Fe-55           [Two-neutron emission, threshold ~11.2 MeV]
  • MT=102  Fe-56(n,γ)Fe-57            [Radiative capture]
  • MT=107  Fe-56(n,α)Cr-53            [Alpha emission (ENDF MT=107)]


In [ ]:
fe56_local = 'jeff_data/n-026_Fe_056.jeff40'

if os.path.exists(fe56_local):
    print(f'✅ Fe-56 file already downloaded → {fe56_local}')
else:
    print('⬇️  Downloading JEFF-4.0 zip from NEA Data Bank...')
    print('   (608 MB — streaming, only extracting Fe-56 file)')

    resp = requests.get(JEFF_ZIP_URL, stream=True, timeout=300)
    resp.raise_for_status()

    total = int(resp.headers.get('content-length', 0))
    buffer = io.BytesIO()

    with tqdm(total=total, unit='B', unit_scale=True, desc='Downloading') as bar:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            buffer.write(chunk)
            bar.update(len(chunk))

    buffer.seek(0)
    print('\n📦 Zip downloaded — extracting Fe-56 file...')

    with zipfile.ZipFile(buffer) as zf:
        all_names = zf.namelist()
        fe56_match = [n for n in all_names if 'Fe_056' in n or 'Fe-056' in n or '026_Fe' in n]
        print(f'   Files matching Fe-56: {fe56_match}')

        if not fe56_match:
            fe_files = [n for n in all_names if 'Fe' in n or '026' in n]
            print(f'   Fe-related files found: {fe_files[:10]}')
            raise FileNotFoundError('Fe-56 file not found in zip — check names above')

        target = fe56_match[0]
        with zf.open(target) as src, open(fe56_local, 'wb') as dst:
            dst.write(src.read())

    size_mb = os.path.getsize(fe56_local) / 1e6
    print(f'✅ Extracted → {fe56_local}  ({size_mb:.1f} MB)')

✅ Fe-56 file already downloaded → jeff_data/n-026_Fe_056.jeff40


In [ ]:
def endf_float(s: str) -> float:
    """
    Parse ENDF compact float notation.
    Examples: '1.00000+0' → 1.0,  '2.56481-3' → 0.00256481
    Also handles standard Python floats and blank fields.
    """
    s = s.strip()
    if not s:
        return 0.0
    try:
        return float(s)
    except ValueError:
        s2 = re.sub(r'([0-9.])([+-])([0-9])', r'\1e\2\3', s)
        return float(s2)


def parse_endf_line(line: str):
    """
    Split one ENDF-6 line into (fields[6], mf, mt).
    """
    line = line.ljust(80)
    data_str = line[:66]
    fields = []
    for i in range(0, 66, 11):
        fld = data_str[i:i+11].strip()
        try:
            fields.append(endf_float(fld) if fld else 0.0)
        except Exception:
            fields.append(0.0)
    try:
        mf = int(line[70:72].strip() or 0)
        mt = int(line[72:75].strip() or 0)
    except ValueError:
        mf, mt = 0, 0
    return fields, mf, mt


def parse_endf_mf3(filepath: str, mt_wanted: set) -> dict:
    """
    Corrected MF=3 parser using an explicit per-section line counter
    instead of an abstract state flag. Returns {mt: DataFrame}.
    """
    results  = {}
    curr_mt  = None
    sec_line = -1       # -1 = not inside a wanted section
    nr = np_ = n_interp_lines = pts_read = 0

    with open(filepath, 'r', errors='replace') as f:
        for raw in f:
            fields, mf, mt = parse_endf_line(raw)

            # SEND (mt=0) or FEND (mf=0): section/file boundary
            if mt == 0 or mf == 0:
                curr_mt  = None
                sec_line = -1
                continue

            if mf != 3:
                continue

            if curr_mt is None:
                if mt in mt_wanted:
                    curr_mt  = mt
                    results.setdefault(mt, [])
                    sec_line = 0      # this line is the HEAD record
                continue

            if mt != curr_mt:
                continue

            sec_line += 1

            if sec_line == 1:
                # CONT record: NR = number of interpolation regions, NP = number of points
                nr             = int(round(fields[4]))
                np_            = int(round(fields[5]))
                n_interp_lines = math.ceil(nr / 3) if nr > 0 else 0
                pts_read       = 0
                continue

            if sec_line <= 1 + n_interp_lines:
                continue   # interpolation table line — skip

            # data line: up to 3 (E, xs) pairs
            for i in range(0, 6, 2):
                if pts_read >= np_:
                    break
                e_eV = fields[i]
                xs   = fields[i + 1]
                if e_eV > 0 or xs != 0:
                    results[curr_mt].append((e_eV * 1e-6, xs))   # eV → MeV
                pts_read += 1

            if pts_read >= np_:
                curr_mt  = None
                sec_line = -1

    dfs = {}
    for mt, pairs in results.items():
        if pairs:
            df = pd.DataFrame(pairs, columns=['energy_MeV', 'cross_section_b'])
            df = df.sort_values('energy_MeV').drop_duplicates().reset_index(drop=True)
            dfs[mt] = df
    return dfs


print('✅ Corrected ENDF-6 parser defined')

✅ Corrected ENDF-6 parser defined


In [ ]:
def _endf_fmt(val):
    if val == 0:
        return '0.000000+0'
    sign = '-' if val < 0 else ' '
    val = abs(val); exp = 0
    while val >= 10: val /= 10; exp += 1
    while val < 1:   val *= 10; exp -= 1
    return f"{sign}{val:.6f}{'+' if exp >= 0 else '-'}{abs(exp)}"

def _endf_line(fields, mat, mf, mt, ln):
    data = ''.join(_endf_fmt(fields[i] if i < len(fields) else 0.0).rjust(11) for i in range(6))
    return f"{data}{mat:4d}{mf:2d}{mt:3d}{ln:5d}"

def _build_section(mat, mt, pairs, regions):
    lines = [_endf_line([26056, 55.5, 0,0,0,0], mat, 3, mt, 1)]
    lines.append(_endf_line([0,0,0,0, len(regions), len(pairs)], mat, 3, mt, 2))
    flat = [v for r in regions for v in r]
    for i in range(0, len(flat), 6):
        lines.append(_endf_line(flat[i:i+6], mat, 3, mt, len(lines)+1))
    flat_d = [v for p in pairs for v in p]
    for i in range(0, len(flat_d), 6):
        lines.append(_endf_line(flat_d[i:i+6], mat, 3, mt, len(lines)+1))
    lines.append(_endf_line([0]*6, mat, 3, 0, 99999))
    return lines

test_lines  = _build_section(2631, 107, [(2e6,1e-5),(4e6,5e-5),(6e6,1e-4),(8e6,2e-4),(1e7,3e-4),(1.5e7,1e-3)], [(2,1),(6,2)])
test_lines += [_endf_line([0]*6, 2631, 0, 0, 0)]

test_path = 'jeff_data/_selftest.tmp'
with open(test_path, 'w') as f:
    f.write('\n'.join(test_lines) + '\n')

test_result = parse_endf_mf3(test_path, {107})
os.remove(test_path)

expected_n = 6
ok = (107 in test_result) and (len(test_result[107]) == expected_n)

if ok:
    print(f'✅ Self-test PASSED — MT=107 (NR=2 case) parsed {len(test_result[107])}/{expected_n} points correctly')
else:
    found = len(test_result.get(107, []))
    print(f'❌ Self-test FAILED — expected {expected_n} points for MT=107, got {found}')
    raise RuntimeError('Parser self-test failed — do not proceed with real data until this passes')

✅ Self-test PASSED — MT=107 (NR=2 case) parsed 6/6 points correctly


In [ ]:
print('🔍 Parsing ENDF-6 file for Fe-56...')
print(f'   Looking for MT sections: {sorted(MT_WANTED)}')

parsed = parse_endf_mf3(fe56_local, MT_WANTED)

print(f'\n✅ Parsing complete — {len(parsed)}/{len(MT_WANTED)} MT sections found:')
for mt in sorted(MT_WANTED):
    if mt in parsed:
        df = parsed[mt]
        emin, emax = df['energy_MeV'].min(), df['energy_MeV'].max()
        print(f'   MT={mt:3d}  {len(df):6d} points  E = {emin:.3e} – {emax:.3e} MeV')
    else:
        print(f'   MT={mt:3d}  ❌ NOT FOUND — check that this MT actually exists in the Fe-56 file')

🔍 Parsing ENDF-6 file for Fe-56...
   Looking for MT sections: [4, 16, 102, 107]

✅ Parsing complete — 3/4 MT sections found:
   MT=  4    2962 points  E = 8.620e-01 – 1.500e+02 MeV
   MT= 16      44 points  E = 1.140e+01 – 1.500e+02 MeV
   MT=102   15151 points  E = 1.000e-11 – 1.500e+02 MeV
   MT=107  ❌ NOT FOUND — check that this MT actually exists in the Fe-56 file


In [ ]:
all_jeff = {}

for rxn in REACTIONS:
    mt = rxn['mt']
    if mt not in parsed:
        print(f"  ⚠️  MT={mt} ({rxn['display']}) — not found in ENDF file, skipping")
        continue

    df = parsed[mt].copy()
    df['reaction'] = rxn['display']
    df['mt']       = mt
    df['source']   = 'JEFF-4.0'
    df['target']   = 'Fe-56'

    path = f"jeff_data/{rxn['label']}_JEFF40.csv"
    df.to_csv(path, index=False)
    all_jeff[rxn['label']] = df
    print(f"  💾 MT={mt:3d}  {rxn['display']:25s}  {len(df):6d} pts → {path}")

print(f'\n✅ {len(all_jeff)}/{len(REACTIONS)} reaction channels saved')
if len(all_jeff) < len(REACTIONS):
    print('⚠️  Not all channels were found — see Cell 7 output above for which MT is missing.')

  💾 MT=  4  Fe-56(n,n')Fe-56*            2962 pts → jeff_data/Fe-56(n,inl)Fe-56star_JEFF40.csv
  💾 MT= 16  Fe-56(n,2n)Fe-55               44 pts → jeff_data/Fe-56(n,2n)Fe-55_JEFF40.csv
  💾 MT=102  Fe-56(n,γ)Fe-57             15151 pts → jeff_data/Fe-56(n,g)Fe-57_JEFF40.csv
  ⚠️  MT=107 (Fe-56(n,α)Cr-53) — not found in ENDF file, skipping

✅ 3/4 reaction channels saved
⚠️  Not all channels were found — see Cell 7 output above for which MT is missing.


In [ ]:
rows = []
for rxn in REACTIONS:
    df = all_jeff.get(rxn['label'], pd.DataFrame())
    if not df.empty:
        emin, emax = df['energy_MeV'].min(), df['energy_MeV'].max()
        xs_max = df['cross_section_b'].max()
        erange = f'{emin:.3e} – {emax:.3e} MeV'
    else:
        erange, xs_max = 'Not found', None
    rows.append({
        'Reaction':       rxn['display'],
        'MT':             rxn['mt'],
        'Data Points':    len(df) if not df.empty else 0,
        'Energy Range':   erange,
        'Peak XS (barn)': f'{xs_max:.4e}' if xs_max else 'N/A',
        'Source':         'JEFF-4.0'
    })

summary = pd.DataFrame(rows)
print('\n📊 JEFF-4.0 Fe-56 Collection Summary')
print('='*95)
print(summary.to_string(index=False))
print('='*95)


📊 JEFF-4.0 Fe-56 Collection Summary
         Reaction  MT  Data Points              Energy Range Peak XS (barn)   Source
Fe-56(n,n')Fe-56*   4         2962 8.620e-01 – 1.500e+02 MeV     1.9990e+00 JEFF-4.0
 Fe-56(n,2n)Fe-55  16           44 1.140e+01 – 1.500e+02 MeV     6.3357e-01 JEFF-4.0
  Fe-56(n,γ)Fe-57 102        15151 1.000e-11 – 1.500e+02 MeV     5.5107e-03 JEFF-4.0
  Fe-56(n,α)Cr-53 107            0                 Not found            N/A JEFF-4.0
